# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates a step-by-step workflow for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined using a Croissant schema. Source schema:

[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`. We use the Croissant schema URL to instantiate the dataset object.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore available record sets, fields, and their unique `@id`s. We'll review what's accessible for tabular analysis.


In [ ]:
# List all record sets and associated field @id's
record_sets = list(dataset.record_sets)

print("Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name})")
    print()

# Optionally, preview the first few records for each record set
for rs in record_sets:
    print(f"Sample records from Record Set `{rs.id}`:")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        if i >= 3:
            break
        print(record)
    print()

## 3. Data Extraction

Load each record set into DataFrames for processing. All entities are referenced by their `@id`.


In [ ]:
# Build DataFrames from each record set
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if len(records) == 0:
        print(f"No records found for `{rs.id}`.")
        continue
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Columns in `{rs.id}`:")
    print(df.columns.tolist())
    print(df.head())

# Select one record set for deeper analysis
selected_record_set_id = None
if len(dataframes):
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Using `{selected_record_set_id}` for subsequent analyses.")

## 4. Exploratory Data Analysis (EDA)

Filter records, normalize numeric fields, and group by categorical values. Use the field `@id`s for all operations.


In [ ]:
# Choose numeric field and group field by @id
df = dataframes[selected_record_set_id]

# Identify numeric fields
numeric_fields = []
for rs in record_sets:
    if rs.id == selected_record_set_id:
        for field in rs.fields:
            # Only consider Integer or Float types
            if hasattr(field, 'data_type') and field.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_fields.append(field.id)

print(f"Numeric fields in `{selected_record_set_id}`: {numeric_fields}")

# For demonstration, pick first numeric field (if available)
if len(numeric_fields) == 0:
    print("No numeric fields found in this record set.")
else:
    numeric_field_id = numeric_fields[0]
    print(f"Analyzing field: {numeric_field_id}")

    # Check for valid numeric values
    numeric_col = numeric_field_id
    df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')

    # Filter records based on threshold
    threshold = 10
    filtered_df = df[df[numeric_col] > threshold]
    print(f"Filtered records where `{numeric_field_id}` > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    if not filtered_df.empty:
        mean_val = filtered_df[numeric_col].mean()
        std_val = filtered_df[numeric_col].std()
        filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - mean_val) / std_val
        print(f"Normalized `{numeric_field_id}` for filtered records:")
        print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

    # Identify all non-numeric fields for grouping
    group_fields = []
    for field in rs.fields:
        if hasattr(field, 'data_type') and field.data_type not in ['schema:Integer', 'schema:Float', 'schema:Number']:
            group_fields.append(field.id)

    if len(group_fields):
        group_field_id = group_fields[0]
        print(f"Grouping by `{group_field_id}`")
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_col].mean().reset_index()
            print(f"Grouped mean of `{numeric_field_id}` by `{group_field_id}`:")
            print(grouped_df.head())
        else:
            print(f"Field `{group_field_id}` not present in DataFrame columns.")

## 5. Visualization

Visualize distributions and relationships among fields using matplotlib and seaborn. All field references are via their `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric field if available
if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of `{numeric_field_id}` in `{selected_record_set_id}`")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot grouped by a non-numeric field
if 'group_field_id' in locals() and group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"Boxplot of `{numeric_field_id}` grouped by `{group_field_id}`")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load a Croissant FAIR² dataset using the `mlcroissant` library.
- Enumerate available record sets, fields, and their `@id`s.
- Extract tabular data from each record set.
- Filter, normalize, and group data using entity `@id`s.
- Visualize distributions and relationships between dataset fields.

For deeper analyses—such as correlating phenotypic data, hormone levels, and genetic variants—extend this workflow by referencing each relevant field or column by its `@id` via the Croissant schema. The small, highly structured scope of the dataset makes it ideal for case studies, clinical illustration and genotype–phenotype investigations, following best practices for reproducibility.
